# Данные Yambda: анализ и подготовка

Посмотрим на журнал, долю прослушивания и события рядом с лайками.
В первой кодовой ячейке `report` показывает сохранённые результаты,
а `recompute` заново готовит данные и строит графики из исходного журнала.


Я выбрал [Yambda flat 50m](https://huggingface.co/datasets/yandex/yambda), потому что в нём есть повторные прослушивания
и явные реакции. Сначала разберусь с данными, затем опишу, как получены метки.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'task_1/src').is_dir() and (path / 'docs/results').is_dir())
TASK = ROOT / 'task_1'
sys.path.insert(0, str(TASK / 'src'))
from notebook_workflow import from_environment

MODE = os.getenv('TLAB_NOTEBOOK_MODE', 'report')  # report или recompute
RUN_NAME = os.getenv('TLAB_NOTEBOOK_RUN', 'repeat_01')
DOWNLOAD_DATA = os.getenv('TLAB_NOTEBOOK_DOWNLOAD', '1') == '1'
INCLUDE_TEST = os.getenv('TLAB_INCLUDE_TEST', '0') == '1'
workflow = from_environment(MODE, RUN_NAME, DOWNLOAD_DATA, INCLUDE_TEST)

workflow.ensure('eda')
EDA = workflow.path('eda')
snapshot = json.loads((EDA / 'snapshot.json').read_text())
print(snapshot['description'])


Сохранённые выводы первичного анализа; новый проход по исходному журналу не выполнялся.


Для просмотра достаточно компактных агрегатов: число событий, частоты
`(played_ratio_pct, is_organic)` и временные гистограммы. Сырых строк здесь нет.
`collect_statistics` в [analyze_dataset.py](../src/analyze_dataset.py)
получает эти таблицы одним проходом по журналу, порциями целых пользователей.
В режиме `recompute` они пересчитываются из исходного parquet;
далее доли и графики строятся теми же ячейками.


## Какие данные есть

In [2]:
sizes = {name: snapshot[name] for name in
         ['rows', 'users', 'items', 'listen_rows', 'reaction_rows', 'duplicated_rows']}
display(pd.Series(sizes, name='Значение').to_frame())
assert sizes['listen_rows'] + sizes['reaction_rows'] == sizes['rows']

shares = pd.DataFrame({
    'События': ['Прослушивания', 'Реакции', 'Точные повторы строк'],
    'Количество': [sizes['listen_rows'], sizes['reaction_rows'], sizes['duplicated_rows']],
})
shares['Доля журнала, %'] = 100 * shares['Количество'] / sizes['rows']
display(shares.round({'Доля журнала, %': 3}))


,Значение
rows,47790449
users,10000
items,934057
listen_rows,46467212
reaction_rows,1323237
duplicated_rows,236230


,События,Количество,"Доля журнала, %"
0,Прослушивания,46467212,97.231
1,Реакции,1323237,2.769
2,Точные повторы строк,236230,0.494


В журнале 47.79 млн строк, из них 46.47 млн прослушиваний. Повторы строк
не удалял: одинаковые записи могут соответствовать разным событиям.
При подготовке сохраняю исходный порядок строк.

In [3]:
schema = pd.read_csv(EDA / 'schema.csv')
display(schema.rename(columns={
    'column': 'Столбец', 'dtype': 'Тип при анализе', 'missing': 'Пропусков',
    'missing_pct': 'Пропусков, %', 'unique': 'Различных значений',
}))

,Столбец,Тип при анализе,Пропусков,"Пропусков, %",Различных значений
0,uid,uint32,0,0.000,10000
1,timestamp,uint32,0,0.000,5052573
2,item_id,uint32,0,0.000,934057
3,is_organic,uint8,0,0.000,2
4,played_ratio_pct,float64,1323237,2.769,160
5,track_length_seconds,float64,1323237,2.769,499
6,event_type,str,0,0.000,5


У реакций нет доли и длины прослушивания. Поэтому пропуски в этих полях
ожидаемы и сами по себе не означают ошибку. В таблице показаны типы из
первичного анализа; при сохранении истории используются более компактные типы.

## Доля прослушивания

In [4]:
ratios = pd.read_csv(EDA / 'ratio_counts.csv')
assert ratios['count'].sum() == snapshot['listen_rows']
assert not ratios.duplicated(['played_ratio_pct', 'is_organic']).any()

# Каждый ряд хранит число событий с данной долей и is_organic.
ratios['Диапазон'] = pd.cut(
    ratios['played_ratio_pct'], [-np.inf, 35, 100, np.inf],
    labels=['Менее 35%', 'От 35 до 99%', 'Не менее 100%'], right=False,
)
ratio_groups = ratios.groupby('Диапазон', observed=False)['count'].sum().to_frame('Прослушиваний')
ratio_groups['Доля, %'] = 100 * ratio_groups['Прослушиваний'] / ratios['count'].sum()
display(ratio_groups.round({'Доля, %': 3}))

bins = np.arange(-0.5, ratios['played_ratio_pct'].max() + 1.5, 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
probabilities = ratios['count'] / ratios['count'].sum()
heights, _ = np.histogram(ratios['played_ratio_pct'], bins=bins, weights=probabilities)
assert np.isclose(heights.sum(), 1)
axes[0].stairs(heights, bins, fill=True, alpha=.7)
axes[0].set(title='Все прослушивания', ylabel='Доля событий')

for organic, group in ratios.groupby('is_organic'):
    probabilities = group['count'] / group['count'].sum()
    heights, _ = np.histogram(group['played_ratio_pct'], bins=bins, weights=probabilities)
    assert np.isclose(heights.sum(), 1)
    axes[1].stairs(heights, bins, label=f'is_organic={organic}')
axes[1].set(title='Распределения внутри is_organic', yscale='log', ylabel='Доля событий, log')
axes[1].legend()
for ax in axes:
    ax.set(xlabel='Доля прослушивания, %')
fig.tight_layout()
plt.show()


,Прослушиваний,"Доля, %"
Диапазон,,
Менее 35%,15827690,34.062
От 35 до 99%,5687263,12.239
Не менее 100%,24952259,53.699


/tmp/ipykernel_1475496/3389841310.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Видно два пика: пользователи часто заканчивают слушать в самом начале
или дослушивают почти весь трек. Одно среднее такую форму описывает плохо.
Порог 35% из этих графиков не следует, это выбранная эвристика разметки.

## Лайк и соседние события

In [5]:
timeline = pd.read_csv(EDA / 'timeline_histograms.csv')
assert (timeline['bin_right'] > timeline['bin_left']).all()
assert (timeline['normalized_value'] >= 0).all()

settings = [
    ('like_seconds', 1, 'Время от первого лайка, секунды'),
    ('like_hours', 3600, 'Время от первого лайка, часы'),
    ('first_listen_days', 86400, 'Время от первого прослушивания, дни'),
]
fig, axes = plt.subplots(3, 1, figsize=(11, 9))
for ax, (histogram, unit, title) in zip(axes, settings):
    selected = timeline.loc[timeline['histogram'].eq(histogram)]
    for event, group in selected.groupby('event'):
        if histogram != 'first_listen_days' and event == 'like':
            continue
        group = group.sort_values('bin_left')
        bins = np.r_[group['bin_left'].to_numpy(), group['bin_right'].iloc[-1]] / unit
        ax.stairs(group['normalized_value'], bins, label=event)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set(xlabel=title, ylabel='Нормированное число событий')
    ax.legend()
axes[0].set_yscale('log')
fig.tight_layout()
plt.show()


/tmp/ipykernel_1475496/304311166.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Рядом с первым лайком пары пользователь–трек много прослушиваний.
На первых двух графиках частоты нормированы на число пар с лайком.
Последний график показывает события относительно первого прослушивания
среди пар с лайком: внутри пользователя пары имеют одинаковый вес,
после чего результат усредняется по пользователям.


## Как получена метка

Готовых меток релевантности 0/1/2 здесь нет, поэтому задал следующие правила
в порядке приоритета:

| label | Правило |
| --- | --- |
| 2 | К прослушиванию привязан like |
| 0 | Иначе: dislike или прослушано менее 35% |
| 1 | Остальные прослушивания |

Реакцию привязываю к ближайшему следующему окончанию той же пары, если она
не раньше восстановленного начала. При like и dislike побеждает like.
Unlike и undislike напрямую метку не задают. Такая метка приближает
релевантность, но не измеряет её напрямую.

In [6]:
rules = pd.DataFrame([
    {'Поле': 'start_ts', 'Смысл': 'floor(timestamp - length * ratio / 100 - 5); только старая привязка реакций'},
    {'Поле': 'timestamp', 'Смысл': 'Исходное время события; точная семантика доступности остаётся допущением'},
    {'Поле': 'artist_id', 'Смысл': 'Первый исполнитель в mapping для трека; неизвестный = -1'},
    {'Поле': 'label', 'Смысл': 'Эвристическая метка результата прослушивания'},
])
display(rules)

,Поле,Смысл
0,start_ts,floor(timestamp - length * ratio / 100 - 5); т...
1,timestamp,Исходное время события; точная семантика досту...
2,artist_id,Первый исполнитель в mapping для трека; неизве...
3,label,Эвристическая метка результата прослушивания


## Почему изменилось время запроса

Восстановленный `start_ts` зависит от текущей доли прослушивания.
Его секундный остаток передавал информацию о метке через свежесть.
Поэтому в основном сравнении время запроса взято из прошлого журнала.
[Диагностика](../../docs/start-time-audit.md).


## Подготовка для моделей

`prepare_history.py` сохраняет полную историю и разметку в listens/events.
`prepare_legacy.py` готовит старые samples и ручные признаки.
Нужные этапы вызываются из ноутбуков в режиме `recompute`.

Дальше — [бейзлайны](model-baseline.ipynb).
[Установка и полный запуск](../../docs/reproduction.md).
